# Provision Spark Connect from Jupyter

This notebook creates a private Spark Connect server on Kubernetes, runs DataFrame and SQL work on executor Pods, reconnects after a real kernel restart, and deletes the recorded resources.

**Run sections 1–6, restart the kernel, then continue at section 7. Do not use Run All across this boundary.** CI executes this exact notebook and performs the kernel restart automatically.

Follow the [Jupyter setup guide](https://hussein.awala.fr/spark-on-k8s/develop/user-guide/jupyter/) to install the client, prepare the namespace/service account, and set explicit connection and image settings. Keep `notebook_support.py` beside this notebook. Python 3.12 is the tested notebook interpreter; CI checks matching client/server pairs for Spark 4.0.5 and 4.3.0. The preflight requires the matching client for this demonstrated workflow; it does not claim that every mixed-version operation is unsupported. The thin client needs no local Spark installation or Java.

The helper owns a loopback-only tunnel to the private server. The SDK does not manage tunnels, authentication gateways, or notebook infrastructure. The small driver/executor resource settings below suit a disposable test cluster.

## 1. Confirm the kernel and connection settings

The artifact directory contains a credential-free server receipt and test evidence. If a receipt exists, reconnect to that server or delete it and confirm cleanup first. Use a fresh directory only when you intentionally need a new server; do not bypass the existing receipt guard.

In [ ]:
from __future__ import annotations

import importlib.metadata as metadata
import json
import os
import sys
from contextlib import ExitStack
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))  # Only the notebook directory, including its companion helper.

import spark_on_k8s
from IPython.display import display
from notebook_support import (
    LoopbackTunnel,
    client_for,
    executor_evidence,
    reserve_receipt,
    settings,
    write_json,
)
from spark_on_k8s.client import ExecutorInstances, PodResources
from spark_on_k8s.connect_server import ConnectServerConfig

cfg = settings()
client = client_for(cfg)
receipt = cfg.artifact_dir / "server-receipt.json"
assert not receipt.exists(), "A receipt exists: resume it below; do not provision a duplicate server."
if os.environ.get("SPARK_NOTEBOOK_EXPECTED_PYTHON"):
    assert Path(sys.executable) == Path(os.environ["SPARK_NOTEBOOK_EXPECTED_PYTHON"])
assert get_ipython().__class__.__name__ == "ZMQInteractiveShell"

if os.environ.get("SPARK_NOTEBOOK_EXPECTED_PYTHON"):
    assert Path(spark_on_k8s.__file__).resolve().is_relative_to(Path(sys.prefix).resolve())
versions = {
    name: metadata.version(name)
    for name in ("spark-on-k8s", "pyspark-client", "ipykernel", "pandas", "pyarrow")
}
if versions["pyspark-client"] != cfg.spark_version:
    raise RuntimeError(
        "This notebook exercises version-sensitive creation of DataFrames from Python data. "
        f"Install pyspark-client=={cfg.spark_version} to match the selected server; "
        f"this kernel has {versions['pyspark-client']}."
    )
first_kernel_pid = os.getpid()
print(
    {
        "python": sys.version.split()[0],
        "kernel": sys.executable,
        "versions": versions,
        "context": cfg.context,
        "namespace": cfg.namespace,
    }
)

## 2. Create the server and persist its identity

The image must contain the matching Spark Connect server JAR. Creation means Kubernetes accepted the resources; readiness still requires a query.

The receipt stores Kubernetes UIDs for safe cleanup. A reservation prevents accidental duplicate creation, and the receipt is published atomically without overwriting an existing receipt. If creation or publication fails, a `.pending` marker blocks retries. Inspect the failure and resources before clearing that marker; absence of a complete receipt does not prove absence of a server. A publication failure prints the full server identity for recovery.

In [ ]:
with reserve_receipt(receipt):
    server = client.create_connect_server(
        ConnectServerConfig(
            image=cfg.image,
            server_jar=cfg.server_jar,
            namespace=cfg.namespace,
            service_account=cfg.service_account,
            app_name="jupyter-connect",
            image_pull_policy=cfg.image_pull_policy,
            driver_resources=PodResources(cpu=1, memory=512, memory_overhead=384),
            executor_resources=PodResources(cpu=1, memory=512, memory_overhead=384),
            executor_instances=ExecutorInstances(initial=1),
            spark_conf={
                "spark.sql.shuffle.partitions": "2",
                "spark.kubernetes.executor.request.cores": "0.25",
            },
        )
    )
    try:
        write_json(receipt, {"schema_version": 1, "server": server.to_dict()}, exclusive=True)
    except BaseException:
        print("Server resources exist; retain this full identity for recovery:")
        print(json.dumps({"schema_version": 1, "server": server.to_dict()}, indent=2))
        raise
    print(
        {
            "server_endpoint": server.endpoint,
            "app_id": server.run.app_id,
            "driver_uid": server.run.driver_uid,
            "receipt": receipt.name,
        }
    )

## 3. Open the local tunnel and verify a real query

The helper waits for Spark's listener before starting a loopback-only <code>kubectl port-forward</code>. Kubernetes Pod Running alone is not query readiness. The SDK then executes an action and checks that the endpoint belongs to the receipt's application.

The tunnel requires access to port-forward and, for the startup check in this example, exec in the driver pod. You can instead manage a tunnel in a separate terminal as described in the project's setup guide.

In [ ]:
tunnel = LoopbackTunnel(cfg, server, "initial")
profile = tunnel.start()
ready = client.wait_connect_server_ready(server, profile=profile, timeout=180)
assert ready["probe"]["app_id"] == server.run.app_id
assert ready["query_ready"] is True
sessions = ExitStack()
spark = sessions.enter_context(profile.session())
assert spark.conf.get("spark.app.id") == server.run.app_id
assert spark.version == cfg.spark_version
print(
    {
        "query_ready": True,
        "remote_spark_version": spark.version,
        "session_type": type(spark).__module__ + "." + type(spark).__name__,
    }
)

## 4. Use DataFrames and SQL, with notebook table rendering

The Python kernel sends the plan and small input data to the remote server. Collect only bounded results into the notebook. Files referenced by Spark live in the server/executor environment, not automatically on your laptop.

In [ ]:
from pyspark.sql import functions as F

distributed_sum = spark.range(0, 10000, 1, 4).agg(F.sum("id").alias("total")).first().total
assert distributed_sum == 49_995_000
events = spark.createDataFrame(
    [("books", 2, 12.5), ("books", 1, 20.0), ("games", 3, 15.0)],
    "category string, quantity long, unit_price double",
)
events.createOrReplaceTempView("notebook_events")
result = spark.sql("""
    SELECT category, SUM(quantity) AS quantity,
           ROUND(SUM(quantity * unit_price), 2) AS revenue
    FROM notebook_events GROUP BY category ORDER BY category
""")
table = result.toPandas()
assert table.to_dict("records") == [
    {"category": "books", "quantity": 3, "revenue": 45.0},
    {"category": "games", "quantity": 3, "revenue": 45.0},
]
display(table)
print("Distributed range sum:", distributed_sum)

In [ ]:
hosts = {
    row.executor_host
    for row in spark.sql(
        "SELECT DISTINCT reflect('java.lang.System', 'getenv', 'HOSTNAME') AS executor_host "
        "FROM range(0, 10000, 1, 4)"
    ).collect()
}
executors = executor_evidence(client, server, hosts)
print(
    {
        "actual_executor_hosts": sorted(hosts),
        "owned_executor_pods": executors,
        "driver_pod": server.run.driver_pod,
    }
)

## 5. Check session boundaries and recover from a bad query

A second Connect session does not inherit the first session's temporary views or SQL settings. Closing it does not stop the shared server. A SQL analysis error also leaves the server available.

In [ ]:
from pyspark.errors import AnalysisException

spark.conf.set("spark.sql.shuffle.partitions", "3")
with profile.session() as second:
    assert not second.catalog.tableExists("notebook_events")
    assert second.conf.get("spark.sql.shuffle.partitions") == "2"
    assert second.range(5).count() == 5
assert spark.sql("SELECT COUNT(*) AS n FROM notebook_events").first().n == 3

try:
    spark.sql("SELECT definitely_missing_notebook_column FROM range(1)").collect()
except AnalysisException as error:
    assert "definitely_missing_notebook_column" in str(error)
    expected_error = type(error).__name__
else:
    raise AssertionError("The intentionally invalid query unexpectedly succeeded")
assert spark.range(7).count() == 7
print({"independent_sessions": True, "expected_query_error": expected_error, "query_after_failure": "passed"})

## 6. Release the client and restart the notebook kernel

This cell closes client sessions and the notebook-owned tunnel. It deliberately leaves the server running. The receipt is the recovery handle if the notebook kernel exits.

After this cell, use **Kernel → Restart Kernel** and continue at section 7. The automated validation restarts the actual IPython kernel at this boundary and verifies that its process ID changed.

In [ ]:
sessions.close()
tunnel.close()
assert client.connect_server_status(server)["status"] == "running"
write_json(
    cfg.artifact_dir / "phase-one.json",
    {
        "kernel_pid": first_kernel_pid,
        "app_id": server.run.app_id,
        "driver_uid": server.run.driver_uid,
        "versions": versions,
        "distributed_sum": distributed_sum,
        "table": table.to_dict("records"),
        "executor_hosts": sorted(hosts),
        "executors": executors,
        "independent_sessions": True,
        "query_error_recovered": True,
        "tunnel_closed": tunnel.process.poll() is not None,
        "server_still_running": True,
    },
)
print("Client sessions and tunnel closed. The server is still running; restart the kernel now.")

## 7. Resume from the receipt in the fresh kernel

This cell imports everything it needs again. It restores the exact recorded server and opens a new tunnel; it does not call the creation API. A restarted kernel creates a new client session, so session-local temporary views must be recreated. Durable tables depend on your separately configured catalog and storage.

In [ ]:
import json
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

from notebook_support import LoopbackTunnel, client_for, settings, wait_for_cleanup, write_json
from spark_on_k8s.connect_server import ConnectServerHandle

cfg = settings()
client = client_for(cfg)
receipt = cfg.artifact_dir / "server-receipt.json"
document = json.loads(receipt.read_text())
assert document["schema_version"] == 1
server = ConnectServerHandle.from_dict(document["server"])
previous = json.loads((cfg.artifact_dir / "phase-one.json").read_text())
assert os.getpid() != previous["kernel_pid"], "Restart the Jupyter kernel before this cell."
if os.environ.get("SPARK_NOTEBOOK_EXPECTED_PYTHON"):
    assert Path(sys.executable) == Path(os.environ["SPARK_NOTEBOOK_EXPECTED_PYTHON"])
assert server.run.app_id == previous["app_id"]
assert server.run.driver_uid == previous["driver_uid"]
tunnel = LoopbackTunnel(cfg, server, "resumed")
profile = tunnel.start()
ready = client.wait_connect_server_ready(server, profile=profile, timeout=180)
assert ready["probe"]["app_id"] == previous["app_id"]
with profile.session() as spark:
    assert not spark.catalog.tableExists("notebook_events")
    reconnect_sum = spark.sql("SELECT SUM(id) AS value FROM range(10000)").first().value
    assert reconnect_sum == previous["distributed_sum"]
print(
    {
        "new_kernel_pid": os.getpid(),
        "same_server_app_id": server.run.app_id,
        "reconnected_query_sum": reconnect_sum,
        "session_temp_view_absent": True,
    }
)

## 8. Delete the server and observe cleanup

Closing the notebook or a Connect session is not server deletion. This final cell explicitly requests UID-checked deletion and then waits for the recorded driver, Services, and owned executor resources to disappear. Repeated deletion is harmless for an absent recorded server.

If execution stops early, use the saved receipt to reconnect or explicitly delete the server; a readiness timeout does not cancel it. The sample tunnel helper is separate from the SDK and should be closed when no longer needed.

In [ ]:
tunnel.close()
deletion = client.delete_connect_server(server)
cleanup = wait_for_cleanup(client, server, [p["uid"] for p in previous["executors"]])
absent = client.delete_connect_server(server)
assert absent["status"] == "absent"
report = {
    "status": "passed",
    "mode": "real-ipykernel-and-kubernetes",
    "spark_version": cfg.spark_version,
    "client_version": previous["versions"]["pyspark-client"],
    "app_id": server.run.app_id,
    "driver_uid": server.run.driver_uid,
    "first_kernel_pid": previous["kernel_pid"],
    "second_kernel_pid": os.getpid(),
    "kernel_restart_verified": True,
    "same_server_reconnected": True,
    "distributed_sum": reconnect_sum,
    "executor_hosts": previous["executor_hosts"],
    "independent_sessions": previous["independent_sessions"],
    "query_error_recovered": previous["query_error_recovered"],
    "deletion": deletion,
    "cleanup": cleanup,
    "repeated_delete": absent,
    "tunnels_closed": previous["tunnel_closed"] and tunnel.process.poll() is not None,
    "versions": previous["versions"],
}
write_json(cfg.artifact_dir / "notebook-result.json", report)
print(
    {
        "status": report["status"],
        "kernel_restart_verified": True,
        "owned_resources_remaining": cleanup["owned_resources_remaining"],
        "repeated_delete": absent["status"],
        "tunnels_closed": report["tunnels_closed"],
    }
)

## Recover after an unexpected kernel exit

Section 7 validates a **planned** restart and uses phase-one.json as test evidence. If the kernel exits earlier, that evidence file may not exist. The actual server lifecycle needs only server-receipt.json; these terminal commands do not depend on notebook state. Use the CLI from your selected notebook environment and the artifact directory configured in the guide:

    spark-on-k8s connect server status "$SPARK_NOTEBOOK_ARTIFACT_DIR/server-receipt.json" --kubeconfig "$SPARK_NOTEBOOK_KUBECONFIG" --context "$SPARK_NOTEBOOK_CONTEXT" --output json
    spark-on-k8s connect server delete "$SPARK_NOTEBOOK_ARTIFACT_DIR/server-receipt.json" --kubeconfig "$SPARK_NOTEBOOK_KUBECONFIG" --context "$SPARK_NOTEBOOK_CONTEXT" --output json

Use the actual artifact-directory path if you overrode it. To resume work instead of deleting, restore ConnectServerHandle from the receipt's inner server mapping, open a fresh tunnel, then use wait_connect_server_ready and a new ConnectProfile session as shown above. The kernel's variables and session-local views do not survive a crash. A notebook-owned tunnel may outlive an abrupt kernel termination; inspect and stop it when no longer needed. The SDK itself does not own tunnels.

## References and boundaries

- [Project server setup guide](https://hussein.awala.fr/spark-on-k8s/develop/user-guide/spark_connect_server/)
- [Spark Connect quickstart](https://spark.apache.org/docs/4.3.0/api/python/getting_started/quickstart_connect.html)
- [Thin Python Spark Connect client installation](https://spark.apache.org/docs/4.3.0/api/python/getting_started/install.html#python-spark-connect-client)
- [IPython kernel installation in an isolated environment](https://ipython.readthedocs.io/en/stable/install/kernel_install.html)
- [Executing notebooks with nbclient](https://nbclient.readthedocs.io/en/latest/client.html)

This validation covers the Python DataFrame/SQL client on the selected Spark version, not every notebook extension, Python UDF dependency, external data connector, or production catalog. Spark Connect does not expose the classic SparkContext/RDD APIs. Client-side files and Python dependencies are not automatically installed on executors.